# Reference-based distribution-shift monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/distribution-shift.ipynb)

Companion notebook for the ReliableML paper `docs/latex/distribution-shift.tex`. It uses `DriftMonitor` (also embedded in `ReliablePipeline`) to compute PSI, KL, KS, Wasserstein, the moment-based DSI and categorical chi-square tests against a fit-time reference.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Statistics

`DriftMonitor.fit` records, per feature, reference quantile bins (numeric) or categories, with extra buckets for underflow/overflow (numeric), unseen categories and missing values. With reference counts $n^{(r)}_b$ and current counts $n^{(c)}_b$ over $B$ buckets, smoothed normalized proportions $p_b,q_b$ (additive $\epsilon=10^{-6}$ per bucket) give
$$\operatorname{PSI}(p,q)=\sum_{b=1}^{B}(q_b-p_b)\log\frac{q_b}{p_b},\qquad D_{\mathrm{KL}}(p\Vert q)=\sum_{b=1}^{B}p_b\log\frac{p_b}{q_b}\ \text{(reference to current)}.$$
Drift flags use PSI against a configurable threshold (`drift_threshold`, default $0.2$) - a convention to be calibrated on your own data, not a universal cutoff.

**Numeric features.** Empirical two-sample KS and 1-D Wasserstein-1:
$$D_{n,m}=\sup_x\lvert\widehat F_{\mathrm{ref}}(x)-\widehat F_{\mathrm{cur}}(x)\rvert,\qquad W_1=\int_{\mathbb R}\lvert\widehat F_{\mathrm{ref}}(x)-\widehat F_{\mathrm{cur}}(x)\rvert\,dx.$$
The library-defined **Drift Score Index** (not a standard test) with mean $\mu$, SD $\sigma$, skewness $\gamma$, excess kurtosis $\kappa$:
$$\operatorname{DSI}^2=\Big(\tfrac{\mu_c-\mu_r}{s}\Big)^2+\Big(\tfrac{\sigma_c-\sigma_r}{s}\Big)^2+(\gamma_c-\gamma_r)^2+(\kappa_c-\kappa_r)^2,\quad s=\max\{\sigma_r,10^{-12}\max(1,\lvert\mu_r\rvert)\}.$$

**Categorical features.** Pearson chi-square homogeneity over reference categories plus an unseen bucket (p-value unreliable for small expected counts).

**Aggregate display** $S_{\mathrm{display}}=\exp(-\overline{\operatorname{PSI}})$ - descriptive, not a probability of correctness. Marginal stability cannot rule out concept drift $P(Y\mid X)$; the separate `assess_concept_shift` needs labels and compares conditional target distributions per feature with a weighted Jensen-Shannon divergence.

## 2. Synthetic reference and a first comparison

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from ml_reliability import DriftMonitor

rng = np.random.default_rng(7)

def make_batch(n, *, shift=0.0, scale=1.0, cat_p=(0.5, 0.3, 0.2), extra_cat=0.0):
    categories = rng.choice(["a", "b", "c"], size=n, p=cat_p)
    if extra_cat:
        categories = np.where(rng.random(n) < extra_cat, "d", categories)  # unseen category
    return pd.DataFrame({
        "x_normal": rng.normal(shift, scale, n),
        "x_skewed": rng.lognormal(0, 0.6, n),
        "category": categories,
    })

reference = make_batch(3000)
monitor = DriftMonitor(n_bins=10, drift_threshold=0.2, bootstrap_samples=50, random_seed=0).fit(reference)

scenarios = {
    "no shift (fresh batch)": make_batch(600),
    "mean +0.5 sd": make_batch(600, shift=0.5),
    "variance x2": make_batch(600, scale=2.0),
    "category mix changed": make_batch(600, cat_p=(0.2, 0.3, 0.5)),
    "unseen category 'd' (15%)": make_batch(600, extra_cat=0.15),
}
reports = {name: monitor.compare(batch) for name, batch in scenarios.items()}
pd.DataFrame([{
    "scenario": name,
    "similarity exp(-mean PSI)": round(r.similarity_score, 3),
    "mean PSI": round(r.mean_psi, 3),
    "covariate shift flagged": r.covariate_shift_detected,
    "PSI-drifted features": ", ".join(r.drifted_features) or "-",
} for name, r in reports.items()])

In [ ]:
def feature_table(report):
    return pd.DataFrame([{
        "feature": f.name, "PSI": f.psi, "KL(ref||cur)": f.kl_divergence, "KS": f.ks_statistic,
        "Wasserstein": f.wasserstein_distance, "DSI": f.dsi, "chi2 p": f.chi_square_p_value,
        "PSI flag": f.drifted, "DSI flag": f.dsi_drifted, "chi2 flag": f.chi_square_drifted,
    } for f in report.features]).round(4)

print("Scenario: mean +0.5 sd")
print(feature_table(reports["mean +0.5 sd"]).to_string(index=False))
print()
print("Scenario: unseen category 'd' (15%)")
print(feature_table(reports["unseen category 'd' (15%)"]).to_string(index=False))
ci = reports["mean +0.5 sd"].features[0].confidence_intervals
print("\nbootstrap CIs for x_normal under +0.5 sd:", {k: tuple(round(v, 3) for v in pair) for k, pair in ci.items()})

## 3. Independent check of the formulas

KS, Wasserstein and the moment score recomputed with SciPy/NumPy, and PSI/KL/chi-square recomputed by hand on the categorical feature, are compared with the report values.

In [ ]:
batch = scenarios["mean +0.5 sd"]
report = reports["mean +0.5 sd"]
f = {x.name: x for x in report.features}

ks = stats.ks_2samp(reference["x_normal"], batch["x_normal"]).statistic
w1 = stats.wasserstein_distance(reference["x_normal"], batch["x_normal"])
def moments(v):
    v = np.asarray(v); return v.mean(), v.std(), stats.skew(v), stats.kurtosis(v)
mr, sr, gr, kr = moments(reference["x_normal"]); mc, sc, gc, kc = moments(batch["x_normal"])
s = max(sr, 1e-12 * max(1, abs(mr)))
dsi = np.sqrt(((mc - mr) / s) ** 2 + ((sc - sr) / s) ** 2 + (gc - gr) ** 2 + (kc - kr) ** 2)
print("KS          report=%.6f  scipy=%.6f" % (f["x_normal"].ks_statistic, ks))
print("Wasserstein report=%.6f  scipy=%.6f" % (f["x_normal"].wasserstein_distance, w1))
print("DSI         report=%.6f  manual=%.6f" % (f["x_normal"].dsi, dsi))

# categorical feature: buckets = a, b, c, unseen, missing; eps = 1e-6 smoothing
cat_report = {x.name: x for x in reports["category mix changed"].features}["category"]
cat_batch = scenarios["category mix changed"]["category"]
ref_counts = np.array([(reference["category"] == c).sum() for c in "abc"] + [0, 0], dtype=float)
cur_counts = np.array([(cat_batch == c).sum() for c in "abc"] + [0, 0], dtype=float)
eps = 1e-6
p = (ref_counts + eps) / (ref_counts.sum() + eps * len(ref_counts))
q = (cur_counts + eps) / (cur_counts.sum() + eps * len(cur_counts))
print("PSI         report=%.6f  manual=%.6f" % (cat_report.psi, np.sum((q - p) * np.log(q / p))))
print("KL(ref||cur) report=%.6f  manual=%.6f" % (cat_report.kl_divergence, np.sum(p * np.log(p / q))))
chi2, pval, _, _ = stats.chi2_contingency(np.vstack([ref_counts[:3], cur_counts[:3]]), correction=False)
print("chi-square  report=%.4f (p=%.3g)  scipy=%.4f (p=%.3g)" % (cat_report.chi_square_statistic, cat_report.chi_square_p_value, chi2, pval))

## 4. Plots: reference vs incoming

In [ ]:
fig, axes = monitor.plot_reference_distributions()
plt.show()
fig, axes = monitor.plot_distributions(scenarios["mean +0.5 sd"], report=reports["mean +0.5 sd"])
plt.show()

## 5. Sensitivity and false alarms

The paper stresses checking thresholds on in-domain batches and on designed shifts. Below, (a) PSI/KS/Wasserstein/DSI of `x_normal` versus the size of a mean shift, and (b) how often fresh in-domain batches trip the PSI flag.

In [ ]:
rows = []
for delta in np.arange(0.0, 2.01, 0.25):
    r = monitor.compare(make_batch(500, shift=delta))
    x = {m.name: m for m in r.features}["x_normal"]
    rows.append({"shift (sd)": delta, "PSI": x.psi, "KS": x.ks_statistic, "Wasserstein": x.wasserstein_distance, "DSI": x.dsi,
                 "similarity": r.similarity_score})
curve = pd.DataFrame(rows)
print(curve.round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(curve["shift (sd)"], curve["PSI"], "o-", label="PSI"); axes[0].axhline(0.2, color="crimson", ls="--", label="threshold 0.2")
axes[0].plot(curve["shift (sd)"], curve["KS"], "s-", label="KS"); axes[0].set_xlabel("mean shift (sd)"); axes[0].legend()
axes[1].plot(curve["shift (sd)"], curve["similarity"], "o-"); axes[1].set_xlabel("mean shift (sd)"); axes[1].set_ylabel("exp(-mean PSI)")
plt.tight_layout(); plt.show()

flags = []
for _ in range(40):
    r = monitor.compare(make_batch(300))
    flags.append(r.covariate_shift_detected)
print("in-domain batches (n=300) flagged as covariate shift: %d of 40" % sum(flags))

## 6. Bundled dataset and covariate vs concept shift

Breast cancer: reference = a training split. A random test batch is compared with a *selected* batch (largest tumours only: covariate shift by selection). Then labels with unchanged inputs are corrupted to show that **concept shift can occur with no covariate shift**.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

bc = load_breast_cancer(as_frame=True)
cols = ["mean radius", "mean texture", "mean smoothness", "mean compactness"]
Xtr, Xte, ytr, yte = train_test_split(bc.data[cols], bc.target, test_size=0.4, stratify=bc.target, random_state=0)
bc_monitor = DriftMonitor(bootstrap_samples=50, concept_shift_threshold=0.1, random_seed=0).fit(Xtr, ytr)

selected = Xte[Xte["mean radius"] > Xte["mean radius"].quantile(0.7)]
for name, batch in [("random test batch", Xte), ("largest 30% by radius", selected)]:
    r = bc_monitor.assess_covariate_shift(batch)
    print(f"{name:24s} n={r.sample_count:3d} similarity={r.similarity_score:.3f} mean PSI={r.mean_psi:.3f} "
          f"flagged={r.covariate_shift_detected} drifted={list(r.drifted_features)}")
fig, axes = bc_monitor.plot_distributions(selected, feature_names=["mean radius", "mean texture"])
plt.show()

flip = np.random.default_rng(0).random(len(yte)) < 0.4
y_corrupted = np.where(flip, 1 - yte.to_numpy(), yte.to_numpy())
for name, labels in [("same labels", yte), ("40% of labels flipped", y_corrupted)]:
    concept = bc_monitor.assess_concept_shift(Xte, labels)
    cov = bc_monitor.assess_covariate_shift(Xte)
    print(f"{name:22s} covariate flagged={cov.covariate_shift_detected} | mean JS={concept.mean_js_divergence:.4f} "
          f"| shifted features={list(concept.shifted_features)}")

## Interpretation and limitations

**How to read the output.** PSI, KL, KS, Wasserstein and DSI measure how far the incoming marginals are from the reference; PSI drives the default flag, DSI and chi-square have their own flags, and `covariate_shift_detected` combines them. Bootstrap intervals describe row-resampling variability. A low shift score does not imply good accuracy and a high one does not imply poor accuracy - check labeled performance when outcomes arrive. The false-alarm count above is an example of calibrating the threshold on in-domain batches.

**Limitations.** Per-feature (marginal) statistics miss shifts that only affect feature interactions; the concept-shift check is univariate and needs labels, so it is retrospective; chi-square p-values need adequate expected counts; the intervals assume independent rows (not clustered or temporal data); the DSI is library-defined, not an inferential test; thresholds must be set with domain data. Reference age, schema, units and batch size all affect the numbers.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.